# Use guard clauses to flatten nested conditions

When a function gathers preconditions before doing its real work, the natural temptation is to nest the work deeper and deeper inside `if` blocks. Guard clauses — early returns or raised exceptions for invalid input — let you keep the main logic at one indentation level. The result is easier to read and easier to extend.

This recipe shows the refactor in three contexts: validation, default handling, and dispatch.

## The shape of the problem

Consider a function that processes an order. Before doing anything, it has to check that the order is valid:

- the order isn't `None`
- the customer is logged in
- the cart isn't empty
- the payment method is set

In [ ]:
def process_order_nested(order):
    if order is not None:
        if order.customer.logged_in:
            if order.cart:
                if order.payment_method:
                    # the actual work, four levels deep
                    total = sum(item.price for item in order.cart)
                    return f"Charged £{total:.2f} via {order.payment_method}"
                else:
                    return "Error: no payment method"
            else:
                return "Error: cart is empty"
        else:
            return "Error: customer not logged in"
    else:
        return "Error: no order supplied"


# Mock objects so we can demonstrate
class Customer:
    def __init__(self, logged_in): self.logged_in = logged_in
class Item:
    def __init__(self, price): self.price = price
class Order:
    def __init__(self, customer, cart, payment_method):
        self.customer = customer
        self.cart = cart
        self.payment_method = payment_method

valid = Order(Customer(True), [Item(9.99), Item(14.50)], "Visa")
print(process_order_nested(valid))
print(process_order_nested(None))
print(process_order_nested(Order(Customer(False), [Item(1)], "Visa")))

The function works, but the actual processing is buried four levels deep, and you have to read every condition before you can find what the function actually does.

## The guard-clause refactor

A guard clause handles the invalid case **and returns immediately**. With each invalid case handled up front, the rest of the function can assume the inputs are good — and stay at one indentation level.

In [ ]:
def process_order_guarded(order):
    if order is None:
        return "Error: no order supplied"
    if not order.customer.logged_in:
        return "Error: customer not logged in"
    if not order.cart:
        return "Error: cart is empty"
    if not order.payment_method:
        return "Error: no payment method"

    # The real work, no longer buried
    total = sum(item.price for item in order.cart)
    return f"Charged £{total:.2f} via {order.payment_method}"


print(process_order_guarded(valid))
print(process_order_guarded(None))
print(process_order_guarded(Order(Customer(False), [Item(1)], "Visa")))

Both versions do the same thing. The guarded version reads as: "Here are the things that would stop us. Here is what we do otherwise." The nested version makes the reader assemble that mental model themselves.

The win compounds as the function grows. Add a fifth check to the nested version and you nest one level deeper. Add it to the guarded version and you add one more `if`-and-`return`.

## Guard with exceptions, not just returns

Sometimes "invalid input" should crash loudly rather than return an error string. The same pattern works with `raise`:

In [ ]:
def process_order_raise(order):
    if order is None:
        raise ValueError("no order supplied")
    if not order.customer.logged_in:
        raise PermissionError("customer not logged in")
    if not order.cart:
        raise ValueError("cart is empty")
    if not order.payment_method:
        raise ValueError("no payment method")

    total = sum(item.price for item in order.cart)
    return f"Charged £{total:.2f} via {order.payment_method}"


print(process_order_raise(valid))

try:
    process_order_raise(None)
except ValueError as e:
    print(f"Caught: {e}")

When the function is called from code that knows what to do with the failure, raising is often cleaner than returning a sentinel — the caller can catch specific exception types and react appropriately.

## Guards for default handling

Guard clauses also clean up "fix the input, then continue" patterns. Instead of nesting the real work inside an `else`:

In [ ]:
# Nested
def greet_nested(name=None):
    if name is None:
        name = "friend"
    else:
        if not name.strip():
            name = "friend"
        else:
            name = name.strip()
    return f"Hello, {name}!"

# Guarded — handle the bad cases, normalise once, then proceed
def greet_guarded(name=None):
    if name is None or not name.strip():
        return "Hello, friend!"
    return f"Hello, {name.strip()}!"

print(greet_nested("  Ada  "))
print(greet_guarded("  Ada  "))
print(greet_nested(""))
print(greet_guarded(""))

## Guards for dispatch

The same pattern works for short-circuit dispatch. Each guard handles one case completely and returns; the function never reaches the bottom unless none of the cases match.

In [ ]:
def describe(value):
    if value is None:
        return "no value"
    if isinstance(value, bool):                    # bool before int!
        return "a yes/no"
    if isinstance(value, (int, float)):
        return f"a number: {value}"
    if isinstance(value, str):
        return f"text of length {len(value)}"
    if isinstance(value, (list, tuple, set)):
        return f"a collection of {len(value)} item(s)"
    return f"some other thing: {type(value).__name__}"

for v in [None, True, 42, 3.14, "hello", [1, 2, 3], {"a": 1}]:
    print(f"{v!r:15} -> {describe(v)}")

(The `isinstance(value, bool)` check has to come before the `int` check because `True` and `False` are technically `int` instances in Python — without that ordering, every `bool` would be reported as "a number".)

## When to keep the nesting

Guard clauses aren't always the right call. Keep the nested form when:

- **The branches are doing different work, not handling errors.** Nesting can be the right structure when there are genuinely two paths through the function.
- **The conditions are interdependent** in a way that flattening obscures. Sometimes `if A and B and C:` reads better than three guards that each only make sense in the context of the others.
- **The function is small.** For a three-line function, the nesting isn't really hurting anyone.

The guard-clause refactor pays off most when a function gathers preconditions before doing real work, and especially when the list of preconditions tends to grow.

## Related reading

- [Avoid common conditional mistakes](avoid-common-conditional-mistakes.md) — including the redundant-`else`-after-`return` pattern that guard clauses naturally eliminate.
- [Choose between if/elif chains, dict dispatch, and match/case](choose-between-conditional-patterns.ipynb) — when guards stop scaling and dispatch tables take over.